# 2026-05-18: 1-turn / 3-turn lookahead AI の 重み NN を REINFORCE 学習

**先頭 cell `LOOKAHEAD_TYPE` を切替えて 2 回 実行**:
1. `LOOKAHEAD_TYPE = 'oneturn'` → `db/snapshots_oneturn.jsonl` 読み、 `weight_nn_oneturn.pt` 出力
2. `LOOKAHEAD_TYPE = 'threeturn'` → `db/snapshots_threeturn.jsonl` 読み、 `weight_nn_threeturn.pt` 出力

REINFORCE 学習: 「勝った試合の重みを NN が出やすく」 update。 from-scratch で 30 epoch。

出力先: `/content/drive/MyDrive/onepiece_nn/` (= ohtsuki さん の Drive)。
完了後 ローカル `db/` に download。

## 0. 学習対象 切替

In [ ]:
# === ここを oneturn / threeturn で 切替 ===
LOOKAHEAD_TYPE = 'oneturn'  # 'oneturn' or 'threeturn'
assert LOOKAHEAD_TYPE in ('oneturn', 'threeturn')
print(f'training NN for: {LOOKAHEAD_TYPE}')

## 1. GPU 確認

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('mem GB:', torch.cuda.get_device_properties(0).total_memory / 1e9)

## 2. Drive マウント + path 設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/onepiece_nn'
SNAPSHOT_PATH = os.path.join(WORK_DIR, f'snapshots_{LOOKAHEAD_TYPE}.jsonl')
OUTPUT_PT = os.path.join(WORK_DIR, f'weight_nn_{LOOKAHEAD_TYPE}.pt')

assert os.path.exists(SNAPSHOT_PATH), f'snapshot 不在: {SNAPSHOT_PATH}'
print(f'snapshot: {SNAPSHOT_PATH} ({os.path.getsize(SNAPSHOT_PATH) // (1024*1024)} MB)')
print(f'output: {OUTPUT_PT}')

## 3. snapshot 読み込み + discount reward 計算

In [ ]:
import json, time
import numpy as np

STATE_DIM = 172
GAMMA = 0.95

t0 = time.time()
snapshots = []
with open(SNAPSHOT_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            snap = json.loads(line)
        except json.JSONDecodeError:
            continue
        if 'state_encoded' not in snap or 'reward' not in snap:
            continue
        snapshots.append(snap)
print(f'loaded {len(snapshots)} snapshots in {time.time()-t0:.1f}s')

from collections import Counter
r_dist = Counter(s['reward'] for s in snapshots)
print(f'reward dist: {dict(r_dist)}')

# discounted reward: 試合終盤に近い snapshot ほど reward 強化
for s in snapshots:
    delta = max(0, s.get('max_turn', s.get('turn', 0)) - s.get('turn', 0))
    s['discounted_reward'] = s['reward'] * (GAMMA ** delta)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_np = np.zeros((len(snapshots), STATE_DIM), dtype=np.float32)
R_np = np.zeros((len(snapshots),), dtype=np.float32)
for i, snap in enumerate(snapshots):
    se = snap['state_encoded']
    if len(se) >= STATE_DIM:
        X_np[i, :] = se[:STATE_DIM]
    else:
        X_np[i, :len(se)] = se
    R_np[i] = snap['discounted_reward']

X = torch.from_numpy(X_np).to(device)
R = torch.from_numpy(R_np).to(device)
print(f'X shape: {X.shape}, R range: [{R.min().item():.2f}, {R.max().item():.2f}]')
print(f'R mean: {R.mean().item():.3f}, std: {R.std().item():.3f}')
del snapshots, X_np, R_np

## 4. WeightNN 定義 (= from-scratch、 base load なし)

In [ ]:
import torch.nn as nn

N_WEIGHTS = 9
BASE_SCALES = torch.tensor([3000, 800, 1500, 1, 500, 1200, 1000, 600, 30000],
                            dtype=torch.float32, device=device)

class WeightNN(nn.Module):
    def __init__(self, input_dim=172, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4), nn.ReLU(),
        )
        self.weight_head = nn.Linear(hidden_dim // 4, N_WEIGHTS)
        self.softplus = nn.Softplus()
        self.register_buffer('base_scales', BASE_SCALES.cpu().clone())
    def forward(self, x):
        h = self.shared(x)
        raw = self.weight_head(h)
        return self.softplus(raw) * self.base_scales

model = WeightNN().to(device)
model.train()
n_params = sum(p.numel() for p in model.parameters())
print(f'from-scratch model, params: {n_params}')

## 5. REINFORCE 学習

In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

EPOCHS = 30
BATCH = 512
LR = 1e-4
NOISE_STD = 0.3

baseline = R.mean()
advantage = R - baseline
print(f'baseline: {baseline.item():.3f}, advantage range: [{advantage.min().item():.2f}, {advantage.max().item():.2f}]')

train_ds = TensorDataset(X, advantage)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)

optimizer = optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    for xb, advb in train_loader:
        optimizer.zero_grad()
        mu = model(xb)
        eps = torch.randn_like(mu) * NOISE_STD * mu.abs().mean(dim=1, keepdim=True)
        sampled = mu + eps
        std = NOISE_STD * mu.abs().mean(dim=1, keepdim=True).clamp(min=1.0)
        log_prob = -0.5 * ((sampled - mu) / std) ** 2
        log_prob = log_prob.sum(dim=1)
        loss = -(log_prob * advb).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    avg = epoch_loss / n_batches
    print(f'  epoch {epoch+1}/{EPOCHS}: avg_loss={avg:.4f}')

torch.save(model.state_dict(), OUTPUT_PT)
print(f'\nsaved: {OUTPUT_PT}')
print(f'→ ローカルに db/weight_nn_{LOOKAHEAD_TYPE}.pt として download してください')